[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/15_samplers_and_solvers.ipynb)

# 15. Samplers and numerical solvers — ODE methods vs diffusion-specific solvers

이전 버전은 DPM-Solver/UniPC를 reference에 적어 놓고 실제로는 일반 Euler/Heun/RK4와 Adams-Bashforth만 구현했다.

이번 버전에서는 먼저 일반 ODE solver를 비교한 뒤, **diffusion의 alpha/sigma/log-SNR 구조를 직접 이용하는 DPM-Solver++ 1차·2차 update**를 실제 식대로 구현한다. UniPC는 generic Adams-Bashforth와 같은 것이 아니므로 별도 predictor-corrector 구조로 구분한다.


In [ ]:
import math

import torch

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. General ODE reference: Euler, Heun, RK4

일반 flow model에서 `dx/dt=f(x,t)`를 적분할 때는 standard ODE method를 그대로 사용할 수 있다. field evaluation 횟수와 truncation error의 trade-off를 먼저 확인한다.


In [ ]:
def field(x, t):
    return -x


x0 = torch.tensor([2.0], device=device)
dt = 0.25
exact = x0 * math.exp(-dt)

euler = x0 + dt * field(x0, 0.0)

k1 = field(x0, 0.0)
euler_prediction = x0 + dt * k1
k2 = field(euler_prediction, dt)
heun = x0 + 0.5 * dt * (k1 + k2)

k1 = field(x0, 0.0)
k2 = field(x0 + 0.5 * dt * k1, 0.5 * dt)
k3 = field(x0 + 0.5 * dt * k2, 0.5 * dt)
k4 = field(x0 + dt * k3, dt)
rk4 = x0 + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6

print("exact:", exact.item())
print("Euler error:", abs(euler - exact).item())
print("Heun error:", abs(heun - exact).item())
print("RK4 error:", abs(rk4 - exact).item())


## 2. DDIM deterministic reconstruction

DDIM에서는 model prediction으로 clean sample `x0_hat` 또는 epsilon을 얻은 뒤 다음 noise level의 `alpha`, `sigma`에 맞춰 sample을 재구성한다. 이것은 generic Euler step과 다른 diffusion schedule-aware update다.


In [ ]:
x_t = torch.tensor([[1.2, -0.7]], device=device)
predicted_x0 = torch.tensor([[1.0, -1.0]], device=device)
predicted_epsilon = torch.tensor([[0.4, 0.6]], device=device)

alpha_next = torch.tensor(0.9, device=device)
sigma_next = torch.sqrt(1 - alpha_next.square())

x_next_ddim = (
    alpha_next * predicted_x0
    + sigma_next * predicted_epsilon
)

print("DDIM-style next sample:", x_next_ddim)


## 3. A tiny VP noise schedule for DPM-Solver++

DPM-Solver 계열의 핵심은 diffusion ODE를 아무 ODE solver에 넣는 것이 아니라 `alpha_t`, `sigma_t`, half-log-SNR `lambda_t = log(alpha_t)-log(sigma_t)`를 이용해 diffusion ODE의 semi-linear structure를 analytically 처리하는 것이다.

아래 schedule은 교육용 cosine VP schedule이며 endpoint singularity를 피하려고 `t∈(0,1)`만 쓴다.


In [ ]:
def alpha(t):
    return torch.cos(0.5 * math.pi * t)


def sigma(t):
    return torch.sin(0.5 * math.pi * t)


def lambda_t(t):
    return torch.log(alpha(t)) - torch.log(sigma(t))


def inverse_lambda(value):
    return (
        2.0 / math.pi
        * torch.atan(torch.exp(-value))
    )


for time in [0.2, 0.5, 0.8]:
    t = torch.tensor(time, device=device)
    print(
        "t=", time,
        "alpha=", float(alpha(t)),
        "sigma=", float(sigma(t)),
        "lambda=", float(lambda_t(t)),
    )


## 4. DPM-Solver++ first-order update

DPM-Solver++는 data prediction `x0_hat`을 사용한다. 한 step update의 핵심은 `sigma_t/sigma_s`와 `expm1(-h)`를 사용해 linear diffusion part를 analytic하게 처리하는 것이다.


In [ ]:
def toy_data_prediction(x, t):
    # 구조 확인용 deterministic model output.
    # 실제 diffusion model에서는 network가 x0 prediction을 반환한다.
    return x / (1 + 0.2 * t)


def dpm_solver_pp_first_order(x_s, s, t, model_fn):
    lambda_s = lambda_t(s)
    lambda_target = lambda_t(t)
    h = lambda_target - lambda_s

    model_s = model_fn(x_s, s)
    phi_1 = torch.expm1(-h)

    x_t = (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * phi_1 * model_s
    )
    return x_t


x_s = torch.tensor([[1.0, -0.5]], device=device)
s = torch.tensor(0.8, device=device)
t = torch.tensor(0.6, device=device)

x_first = dpm_solver_pp_first_order(
    x_s,
    s,
    t,
    toy_data_prediction,
)

print("DPM-Solver++ first-order:", x_first)


## 5. DPM-Solver++ second-order single-step update

2차 update는 log-SNR interval 내부의 midpoint `s1`에서 model을 한 번 더 평가한다. 두 model prediction의 차이로 first-order error를 보정한다.


In [ ]:
def dpm_solver_pp_second_order(
    x_s,
    s,
    t,
    model_fn,
    r1=0.5,
):
    lambda_s = lambda_t(s)
    lambda_target = lambda_t(t)
    h = lambda_target - lambda_s

    lambda_s1 = lambda_s + r1 * h
    s1 = inverse_lambda(lambda_s1)

    model_s = model_fn(x_s, s)

    phi_11 = torch.expm1(-r1 * h)
    x_s1 = (
        sigma(s1) / sigma(s) * x_s
        - alpha(s1) * phi_11 * model_s
    )

    model_s1 = model_fn(x_s1, s1)
    phi_1 = torch.expm1(-h)

    correction = (
        0.5 / r1
        * alpha(t)
        * phi_1
        * (model_s1 - model_s)
    )

    x_t = (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * phi_1 * model_s
        - correction
    )

    return x_t, s1


x_second, midpoint_time = dpm_solver_pp_second_order(
    x_s,
    s,
    t,
    toy_data_prediction,
)

print("midpoint time:", float(midpoint_time))
print("DPM-Solver++ second-order:", x_second)


## 6. Multistep history is not merely Adams-Bashforth

DPM-Solver++ multistep과 UniPC는 이전 model evaluations를 재사용하지만, diffusion schedule/log-SNR에 맞는 coefficient를 사용한다. 아래 generic AB2는 history reuse의 baseline일 뿐이며 **DPM-Solver++/UniPC 구현이라고 부르지 않는다**.


In [ ]:
x = torch.tensor([2.0], device=device)
previous_field = torch.tensor([-2.2], device=device)
current_field = field(x, 0.0)

ab2 = x + dt * (
    1.5 * current_field
    - 0.5 * previous_field
)

print("generic AB2 baseline:", ab2)


## 7. Predictor-corrector structure used by UniPC

UniPC의 중요한 구조는 **UniP predictor가 다음 sample을 예측하고, 새 시점의 model evaluation을 얻은 뒤 UniC corrector가 같은 analytical form 안에서 보정**한다는 점이다. 아래 코드는 그 control flow를 명시적으로 보여주되, 논문의 arbitrary-order coefficient solver 전체를 작은 scalar 예제로 위조하지 않는다.


In [ ]:
def predictor_corrector_step(x, t, dt, model_fn):
    model_now = model_fn(x, t)

    predicted = x + dt * model_now
    next_t = t + dt
    model_next = model_fn(predicted, next_t)

    corrected = x + 0.5 * dt * (model_now + model_next)
    return predicted, corrected


def simple_model(x, t):
    return -x + 0.1 * t


predicted, corrected = predictor_corrector_step(
    torch.tensor([1.0], device=device),
    torch.tensor(0.0, device=device),
    0.2,
    simple_model,
)

print("predictor output:", predicted)
print("corrector output:", corrected)


## References and provenance

**DDIM** — Song et al., deterministic diffusion update를 참조했다.

**DPM-Solver / DPM-Solver++** — Lu et al. 및 공식 `LuChengTHU/dpm-solver` 구현. diffusion half-log-SNR `lambda`, `expm1`, data prediction 기반 first-order와 second-order single-step update를 반영했다.

**UniPC** — Zhao et al., *UniPC: A Unified Predictor-Corrector Framework for Fast Sampling of Diffusion Models*. UniP predictor + 새 model evaluation + UniC corrector 구조를 구분했다. 이 노트북의 마지막 predictor-corrector cell은 control-flow illustration이며 arbitrary-order UniPC coefficient solver 전체 구현은 아니다.

**Euler / Heun / RK4 / Adams-Bashforth** — standard numerical ODE baselines. diffusion-specific solver와 혼동하지 않는다.
